# Multi-view reconstruction from a CZI file

This notebook shows an end-to-end `multiview-stitcher` workflow for a rotated multi-view light-sheet dataset acquired on a Zeiss Lightsheet microscope:

1. **Load** the views from the CZI file
2. **Convert** them to OME-Zarr
3. **Register** the views — either bead-based (default) or intensity-based
4. **Fuse** the registered views, using weighted average fusion and multi-view deconvolution

## Dataset

We use the `Dmel_btd-gap_1tp_5v_2c_beads.czi` sample (1 timepoint, 5 views, 2 channels, with fiducial beads) from the [2026 light-sheet image analysis workshop](https://bruvellu.github.io/light-sheet-image-analysis-workshop-2026/practicals/practical_multiview/), which builds on the BigStitcher tutorial dataset. Download the file from that page before running the notebook.

Optional dependencies used below:

- intensity-based registration: `pip install multiview-stitcher[itk-elastix]`
- GPU-accelerated deconvolution: `cupy`

In [ ]:
from pathlib import Path

from dask.diagnostics import ProgressBar
from tqdm.auto import tqdm

from multiview_stitcher import (
    czi_utils,
    detection,
    fusion,
    misc_utils,
    msi_utils,
    ngff_utils,
    registration,
    vis_utils,
)
from multiview_stitcher import spatial_image_utils as si_utils

%matplotlib ipympl

## 1. Load the views

`czi_utils.read_multiview_czi_into_sims` lazily reads every view of a multi-view CZI file into a spatial image (`sim`), with pixel spacing and channel names taken from the file metadata. The rotation of each view around the sample axis is read from the metadata as well and attached to the sim as an affine transformation under the transform key `"metadata"`.

Nothing is read from disk yet — the sims are backed by dask arrays.

*Tip: if the views end up rotated in the wrong direction, pass `invert_angles=True`. The sign convention of the rotation angles is not consistent across CZI files.*

In [ ]:
# adapt this path to where you downloaded the dataset
# fn = Path("../image-datasets/drosophila_czi/Dmel_btd-gap_1tp_5v_2c_beads.czi")
fn = Path("/links/shared/scuanalysis/SCU/Marvin/multiview-stitcher/image-datasets/drosophila_czi/Dmel_btd-gap_1tp_5v_2c_beads.czi")

sims = czi_utils.read_multiview_czi_into_sims(str(fn))
msims = [msi_utils.get_msim_from_sim(sim) for sim in sims]

print(f"Loaded {len(sims)} views")
sims[0]

`vis_utils.plot_positions` shows how the views are placed relative to each other in the coordinate system defined by a transform key — here the microscope metadata. This is the starting point that registration will refine.

In [ ]:
vis_utils.plot_positions(msims, transform_key="metadata")

## 2. Convert the views to OME-Zarr

Reading planes directly from the CZI file is slow and does not support chunked, parallel access. We therefore write each view to OME-Zarr once and read it back as a multiscale image (`msim`). The multiple resolution levels also speed up visualization and registration.

OME-Zarr cannot yet store affine transformations, so we re-attach the metadata transform of each view after reading it back.

In [ ]:
ome_zarr_dir = fn.parent / "ome_zarr"
ome_zarr_dir.mkdir(parents=True, exist_ok=True)
ome_zarr_paths = [ome_zarr_dir / f"view_{iview}.ome.zarr" for iview in range(len(sims))]

for sim, path in zip(tqdm(sims, desc="Writing OME-Zarr"), ome_zarr_paths):
    ngff_utils.write_sim_to_ome_zarr(
        sim.chunk({dim: 1 if dim in ["t", "c"] else 256 for dim in sim.dims}),
        str(path),
        overwrite=True,
        batch_options={
            "batch_func": misc_utils.process_batch_using_joblib,
            "batch_func_kwargs": {"n_jobs": 4},
            "n_batch": 20,
        },
    )

# read back as msims and re-attach the metadata transform
msims = [ngff_utils.read_msim_from_ome_zarr(str(path)) for path in ome_zarr_paths]
for sim, msim in zip(sims, msims):
    msi_utils.set_affine_transform(
        msim,
        si_utils.get_affine_from_sim(sim, transform_key="metadata"),
        transform_key="metadata",
    )

msims[0]

## 3. Inspect the views before registration

`vis_utils.view_neuroglancer` opens the views in a browser-based [neuroglancer](https://neuroglancer-docs.web.app/) viewer, applying the affine transformations associated with the given transform key. Here we look at the views as positioned by the microscope metadata — they should roughly overlap, but not yet accurately.

The image data is streamed from the OME-Zarr files on disk, while the transformations come from the in-memory msims.

*Interrupt the notebook cell to stop the viewer.*

In [ ]:
channels = [str(channel) for channel in sims[0].coords["c"].values]
print("Channels:", channels)

viz_channel = channels[1]
print("Chosen channel for visualization:", viz_channel)

vis_utils.view_neuroglancer(
    # ome_zarr_paths=[str(path) for path in ome_zarr_paths],
    images=[msi_utils.multiscale_sel_coords(msim, {"c": viz_channel}) for msim in msims],
    transform_key="metadata",
    contrast_limits=(0, 3000),
    use_positional_colors=True
)

## 4. Registration

`registration.register` builds a graph of overlapping views, runs a **pairwise registration** for each pair, and then finds a globally consistent set of transformations (**global parameter resolution**), which it attaches to the msims under `new_transform_key`.

Below are two **alternative** ways of registering this dataset:

- **Option A (default): bead-based** — detect the fiducial beads in each view and match them between views. Fast and robust, and the recommended path for this dataset.
- **Option B: intensity-based** — align the image content itself, first with phase correlation, then refined with elastix. Use this when no beads are available.

Both write their result to the same transform key `"registered"`, so everything below the registration section works the same either way. **Run one of the two options, not both** — the second one you run would overwrite the result of the first.

Both options work on a single channel, which we select first. This dataset images the beads in a dedicated channel, which is the one to register on in either case.

In [ ]:
channels = [str(channel) for channel in sims[0].coords["c"].values]
print("Channels:", channels)

reg_channel = channels[1]
print("Chosen channel for registration:", reg_channel)

### Option A (default): bead-based registration

First we detect the beads. `detection.detect_beads` runs a detection function chunk-wise over a view and returns the detected positions in physical coordinates. `detection.log_detect` is a Laplacian-of-Gaussian detector whose main parameter is the physical size of the beads.

We attach the resulting point set to each msim under the points key `"beads"`.

In [ ]:
for msim in tqdm(msims, desc="Detecting beads"):
    beads = detection.detect_beads(
        msi_utils.multiscale_sel_coords(msim, sel_dict={"c": reg_channel}),
        detection_func=detection.log_detect,
        detection_func_kwargs={
            "target_size_physical": 2.0,  # approximate bead diameter in µm
            "threshold_abs": 1000,  # minimum bead intensity
            "max_neigh_intensity": 500,  # reject beads sitting on bright background
            "max_neigh_sigma": 1,
        },
    )
    msi_utils.set_point_set(msim, beads, points_key="beads")

print("Detected beads per view:", [msi_utils.get_point_set(msim).sizes["point_id"] for msim in msims])

It is worth checking the detections before registering: `vis_utils.imshow` overlays the point set on a projection of the view.

In [ ]:
vis_utils.imshow(
    msi_utils.multiscale_sel_coords(msims[0], {"c": reg_channel}),
    points_key="beads",
    points_tolerance=3, # maximum distance in pixels to show a point in z
    resolution_level=2,
    imshow_kwargs={"vmin": 0, "vmax": 500},
    figure_kwargs={"figsize": (15, 10)},
)

Now we register. Passing `points_key="beads"` together with `pairwise_reg_func=registration.registration_marker_based` makes `register` match the detected bead constellations between overlapping views instead of comparing image intensities. The matching is descriptor-based and followed by a RANSAC fit, in the spirit of BigStitcher's bead-based registration.

Since only point coordinates are compared, this is cheap — no image data is loaded for the pairwise registrations.

`plot_summary=True` displays diagnostics for both the pairwise registrations and the global parameter resolution.

In [ ]:
params = registration.register(
    msims,
    transform_key="metadata",  # start from the microscope metadata transform
    new_transform_key="registered",  # store the result here
    reg_channel=reg_channel,
    points_key="beads",  # match detected beads instead of image intensities
    pairwise_reg_func=registration.registration_marker_based,
    pairwise_reg_func_kwargs={
        "transform_type": "rigid",  # solve rotation + translation between views
        "ransac_max_error": 10,  # maximal residual (in µm) for an inlier match
        "descriptor_ratio": 2,  # ratio test used during descriptor matching
        "ransac_min_inlier_ratio": 0.01,  # tolerate sparse but valid bead matches
        "ransac_min_inlier_factor": 1,
        "random_state": 0,  # keep the example deterministic
    },
    groupwise_resolution_kwargs={"transform": "rigid"},
    pre_registration_pruning_method=None,  # keep all overlapping pairs
    plot_summary=True,
)

params

### Option B (alternative): intensity-based registration

Use this path if your dataset contains no beads. It runs in two steps:

1. **Phase correlation** to obtain a translational alignment, stored under `"translation_registered"`.
2. **Elastix** (`registration.registration_ITKElastix`, requires `itk-elastix`) starting from that result, refining it to a full affine transformation stored under `"registered"`.

Because the views of this dataset are arranged in a ring around the rotation axis, we register consecutive views explicitly via `pairs` instead of relying on the automatic overlap graph.

Registering on image intensities is considerably more expensive than matching beads, so we bin the data in `y` and `x` beforehand.

⚠️ Skip this section if you ran Option A above.

In [ ]:
n_views = len(msims)
view_pairs = [(iview, iview + 1) for iview in range(n_views - 1)] + [(n_views - 1, 0)]

with ProgressBar():
    registration.register(
        msims,
        transform_key="metadata",
        new_transform_key="translation_registered",
        reg_channel=reg_channel,  # channel to use for registration
        registration_binning={"z": 1, "y": 4, "x": 4},
        pairs=view_pairs,
        groupwise_resolution_kwargs={"transform": "translation"},
        n_parallel_pairwise_regs=2,  # lower this further if memory is limited
        plot_summary=True,
        pre_registration_pruning_method=None,
        # alternatively, only register views that maximally differ by 90 degrees in rotation:
        # pre_registration_pruning_method="keep_axis_aligned",
        # pre_reg_pruning_method_kwargs={
        #     "max_angle": 90 * 3.14159 / 180,  # in radians
        # },
    )

In [ ]:
with ProgressBar():
    registration.register(
        msims,
        transform_key="translation_registered",  # start from the phase correlation result
        new_transform_key="registered",
        reg_channel=reg_channel,
        registration_binning={"z": 1, "y": 4, "x": 4},
        pairwise_reg_func=registration.registration_ITKElastix,
        pairwise_reg_func_kwargs={
            "transform_types": ["Translation", "Rigid", "Affine"],
            "number_of_resolutions": 3,
            "number_of_iterations": 500,
        },
        pairs=view_pairs,
        groupwise_resolution_kwargs={"transform": "affine"},
        n_parallel_pairwise_regs=2,
        plot_summary=True,
        pre_registration_pruning_method=None,
        # alternatively, only register views that maximally differ by 90 degrees in rotation:
        # pre_registration_pruning_method="keep_axis_aligned",
        # pre_reg_pruning_method_kwargs={
        #     "max_angle": 90 * 3.14159 / 180,  # in radians
        # },
    )

## 5. Inspect the views after registration

Same visualizations as before registration, now using the `"registered"` transform key. The views should now be well aligned — a good place to judge registration quality before spending time on fusion.

*Interrupt the notebook cell to stop the viewer.*

In [ ]:
vis_utils.plot_positions(msims, transform_key="registered")

In [ ]:
vis_utils.view_neuroglancer(
    images=[msi_utils.multiscale_sel_coords(msim, {"c": reg_channel}) for msim in msims],
    transform_key="registered",
    contrast_limits=(0, 3000),
    use_positional_colors=True
)

## 6. Fusion

`fusion.fuse` transforms all views into a common output coordinate system and combines them into a single image. By default, overlapping views are combined by weighted average, with `blending_widths` controlling how smoothly the views blend into each other.

The result is streamed directly into an OME-Zarr file, so datasets larger than memory can be fused as well.

In [ ]:
output_spacing = {"z": 0.5, "y": 0.5, "x": 0.5}
blending_widths = {"z": 50, "y": 50, "x": 50}

fused = fusion.fuse(
    images=msims,
    transform_key="registered",
    output_spacing=output_spacing,
    blending_widths=blending_widths,
    output_chunksize=256,
)

fused

In [ ]:
output_spacing = {"z": 0.5, "y": 0.5, "x": 0.5}
blending_widths = {"z": 50, "y": 50, "x": 50}

fused = fusion.fuse(
    images=msims,
    transform_key="registered",
    output_spacing=output_spacing,
    blending_widths=blending_widths,
    output_chunksize=256,
    output_zarr_url=str(fn.parent / "fused_weighted_average.ome.zarr"),
    zarr_options={"ome_zarr": True, "overwrite": True},
    batch_options={
        "batch_func": misc_utils.process_batch_using_joblib,
        "batch_func_kwargs": {"n_jobs": 4}, # parallelize across 4 jobs
        "n_batch": 10, # number of batches to split the fusion into
    },
)

fused

## 7. Multi-view deconvolution

Instead of averaging the views, they can be combined by multi-view deconvolution, which additionally deconvolves the (view-dependent) PSFs and yields a more isotropic result. This is done by passing `fusion.multi_view_deconvolution` as the `fusion_func` of `fusion.fuse`.

The PSFs are modelled from the given numerical aperture and emission wavelength. This step is considerably more expensive than weighted average fusion — pass `backend="cupy"` to run it on the GPU.

In [ ]:
deconvolved = fusion.fuse(
    images=msims,
    transform_key="registered",
    output_spacing=output_spacing,
    blending_widths=blending_widths,
    output_chunksize=256,
    fusion_func=fusion.multi_view_deconvolution,
    fusion_func_kwargs={
        "n_iterations": 10,
        "output_spacing": output_spacing,
        "na": 0.8,  # numerical aperture of the detection objective
        "wavelength_um": 0.52,  # emission wavelength
    },
    # backend="cupy",  # uncomment to run the deconvolution on the GPU
    output_zarr_url=str(fn.parent / "fused_deconvolved.ome.zarr"),
    zarr_options={"ome_zarr": True, "overwrite": True},
    batch_options={
        "batch_func": misc_utils.process_batch_using_joblib,
        "batch_func_kwargs": {"n_jobs": 4}, # parallelize across 4 jobs
        "n_batch": 10, # number of batches to split the fusion into
    },
)

deconvolved

Finally, we can visualize the result:

In [ ]:
vis_utils.view_neuroglancer(
    images=[fused], # or [deconvolved] to visualize the deconvolved result
    transform_key="registered",
    contrast_limits=(0, 3000),
)